# Transfer learning of the temporal side: other cities' PM2.5 → Manila

The CNN branch adds no accuracy (four-model comparison, learning curve), so transfer learning goes where the skill is: the **temporal branch**. An LSTM-only model is
**pretrained on other Southeast Asian cities' hourly PM2.5** (`data/Other Countries/`, from `other-country-scrapers/scrapers.ipynb`) and then **fine-tuned on Manila**. No weather, no morphology.
Everything is scored on the **same Manila test stations**, so the numbers are directly comparable.

**The checkpoints compared** (`GROUPS = "auto"` builds them from the folders it finds)

| name | pretrained on |
|---|---|
| `manila_only` | nothing: the baseline, trained from scratch |
| `single_<country>` | the cities of one country (one run per country: the cleanest control, shows which country transfers) |
| `mmd_top_k` | all countries, but only the `TOP_K` stations whose PM2.5 distribution is closest to the Manila training stations (maximum mean discrepancy) |
| `all_sources` | every source station, unfiltered (shows negative transfer if there is any) |

**How each checkpoint is used** (`VARIANTS`)

| variant | what it does |
|---|---|
| `finetune` | start from the pretrained weights, train everything on Manila (the usual transfer) |
| `head_only` | the pretrained LSTM stays frozen, only the fusion head is trained on Manila |
| `zero_shot` | the pretrained weights as they are (one pass with learning rate 0): how well Manila is predicted without seeing it |

Judge a checkpoint by `finetune` against `manila_only`, by cross-validation R² when `USE_CV = True` (the default; about 4x the time of a plain fit, still minutes for an LSTM-only model); `zero_shot` says how much transfers without any Manila training. The cost is small: an LSTM-only fit takes minutes.
Same resume and time budget as the other notebooks (`TIME_BUDGET_H`; every finished step is skipped on a re-run).

| Cell | What it does |
|---|---|
| 1 | settings |
| 2 | helpers and checks |
| 3 | gets the repository, the data (including `Other Countries`), the libraries (Kaggle only) |
| 4 | preprocess Manila, pretrain each checkpoint, fine-tune and report each variant |
| 5 | the comparison table and figure |

## 1. Settings (the only cell you edit)

In [ ]:
from pathlib import Path

ARM = "from_2023_with_preinstall"      # the Manila dataset: data/Combined Dataset/<ARM>
PIN_PARTITION = "model/configs/pinned_partition_2026-11.json"   # the test (and calibration) stations of every result so far; None = let the seed pick them again
RASTER = "master_raster_v3.tif"        # needed to preprocess Manila (stations, spatial split); the source cities use no raster

GROUPS = "auto"                         # "auto": manila_only + one single_<country> per country folder + mmd_top_k + all_sources.
                                        # Or your own: {"manila_only": [], "vn_only": ["vietnam/hanoi"], "th_id": ["thailand/bangkok", "indonesia/jakarta"]}
                                        # (a name -> the source cities, as folders under data/Other Countries). [] means no pretraining
TOP_K = 10                              # for "mmd_top_k": the number of source stations kept (the closest to the Manila training stations)
EXCLUDE_MONTHS = []                     # calendar months (1-12) left out of the SOURCE series, e.g. [2, 3, 4] for the regional burning season. [] = none: test it, do not assume it
VARIANTS = ["finetune", "head_only", "zero_shot"]     # how each pretrained checkpoint is used (see above); "manila_only" is always the plain fit
USE_CV = True                           # True: pools are ranked by SPATIAL CROSS-VALIDATION on the Manila training stations (the test stations play no part in the choice); about 4x the time.
                                        # False: no cross-validation, fixed EPOCHS, pools compared on the test stations directly (quick, but the winner's test score is then a little optimistic)
EPOCHS = 15                             # epochs of every Manila fit when USE_CV is False (what cross-validation chose for the LSTM-only model); with USE_CV the fold early stopping picks them
PRETRAIN_EPOCHS = 30                    # upper bound for the pretraining (early stopping on held-out source stations ends it sooner)
TIME_BUDGET_H = 11.0                    # stop by itself after this many hours (Kaggle ends a session at 12). Save Version, add it as an Input, run again to continue
REDO = []                               # steps to run again although their output exists: any of "preprocess", "pretrain", "train", "report"
                                        # (redoing "preprocess" redoes all the later steps: a model only fits the data it was trained on)

DATA = Path("/kaggle/input/datasets/patrickhansperez/urban-air-data")                    # your data dataset: the exact path shown in the Input panel
GITHUB = "https://github.com/Patthethicc/urban-air.git"        # the code, used when no saved repository is attached
REFRESH_CODE = False                    # True: after restoring a saved repository, replace its code (not its runs) by the latest from GITHUB

## 2. Helpers and checks

In [ ]:
import json, os, shutil, subprocess, sys, time, traceback
import numpy as np, pandas as pd

T0 = time.time()                         # the clock of the time budget starts here (cell 1); a new session starts it again
ON_KAGGLE = Path("/kaggle/working").exists()
INPUT = Path("/kaggle/input")
if ON_KAGGLE:
    REPO = Path("/kaggle/working/urban-air")
else:                                    # a local checkout: open the notebook from the repository root
    REPO = Path.cwd() if (Path.cwd() / "model").is_dir() else Path.cwd().parent
MODEL = REPO / "model"
OUT = Path("/kaggle/working") if ON_KAGGLE else REPO       # where the comparison table and figure are written
STEPS = ("preprocess", "pretrain", "train", "report")
MIN_STEP_H = 0.1                         # a step is not started with less than this left of the time budget
BASE = f"runs/{ARM}/tl_cv" if USE_CV else f"runs/{ARM}/tl"      # the Manila fits of this notebook (separate folders: a run without cross-validation is never taken for one with it)
PRETRAINED = f"runs/{ARM}/tl/pretrained"                       # <PRETRAINED>/<checkpoint name>/pretrained_lstm.pt (pretraining never sees Manila, so both modes share it)


class OutOfTime(Exception):
    """The time budget is used up: stop cleanly so that Save Version keeps everything."""


def need(*names):
    """A clear message instead of a NameError at the end of a long run: the names a cell relies on must exist."""
    missing = [n for n in names if n not in globals()]
    if missing:
        raise RuntimeError(f"{', '.join(missing)} not defined: run the cells above in order (1 settings, 2 helpers, 3 setup) first")


def source_cities():
    """The city folders under data/Other Countries (relative names like 'vietnam/hanoi') that hold a source.json and sensor files."""
    root = REPO / "data" / "Other Countries"
    return sorted(p.parent.relative_to(root).as_posix() for p in root.rglob("source.json") if any((p.parent / "sensors").glob("*.csv"))) if root.exists() else []


def groups():
    """name -> source cities, from GROUPS ('auto' builds it from the folders found). The top-k group is marked with the number of stations."""
    if GROUPS != "auto":
        return dict(GROUPS)
    cities = source_cities()
    out = {"manila_only": []}
    for country in sorted({c.split("/")[0] for c in cities}):
        out[f"single_{country}"] = [c for c in cities if c.split("/")[0] == country]
    if cities:
        out[f"mmd_top_{TOP_K}"] = list(cities)
        out["all_sources"] = list(cities)
    return out


def check_settings():
    need("ARM", "RASTER", "USE_CV", "GROUPS", "TOP_K", "EXCLUDE_MONTHS", "VARIANTS", "EPOCHS", "PRETRAIN_EPOCHS", "TIME_BUDGET_H", "REDO", "DATA", "GITHUB", "REFRESH_CODE")
    global REDO
    if GROUPS != "auto" and (not isinstance(GROUPS, dict) or not GROUPS):
        raise ValueError(f'GROUPS must be "auto" or a non-empty dict name -> list of source cities, got {GROUPS!r}')
    bad = [v for v in VARIANTS if v not in ("finetune", "head_only", "zero_shot")]
    if bad or not VARIANTS:
        raise ValueError(f"VARIANTS must be some of finetune, head_only, zero_shot, got {VARIANTS!r}")
    if not isinstance(TOP_K, int) or TOP_K < 1:
        raise ValueError(f"TOP_K must be a whole number of 1 or more, got {TOP_K!r}")
    if [m for m in EXCLUDE_MONTHS if not isinstance(m, int) or not 1 <= m <= 12]:
        raise ValueError(f"EXCLUDE_MONTHS must be calendar months 1-12, got {EXCLUDE_MONTHS!r}")
    if not isinstance(EPOCHS, int) or EPOCHS < 1 or not isinstance(PRETRAIN_EPOCHS, int) or PRETRAIN_EPOCHS < 1:
        raise ValueError("EPOCHS and PRETRAIN_EPOCHS must be whole numbers of 1 or more")
    if not isinstance(USE_CV, bool):
        raise ValueError(f"USE_CV must be True or False, got {USE_CV!r}")
    if not TIME_BUDGET_H > 0:
        raise ValueError(f"TIME_BUDGET_H must be above 0, got {TIME_BUDGET_H!r}")
    unknown = [r for r in REDO if r not in STEPS]
    if unknown:
        raise ValueError(f"REDO has unknown steps {unknown}; use any of {STEPS}")
    if "preprocess" in REDO:             # a model only fits the data it was trained on: redo everything after it
        REDO = sorted(set(REDO) | set(STEPS))


def left_h():
    """Hours left of the time budget."""
    return TIME_BUDGET_H - (time.time() - T0) / 3600


def jobs():
    """(checkpoint name, variant) in the order they run: the Manila-only baseline first, then every checkpoint with each variant."""
    out = []
    for name, cities in groups().items():
        if not cities:
            out.append((name, "scratch"))                 # no pretraining: the plain Manila fit
        else:
            out += [(name, v) for v in VARIANTS]
    return out


def job_dir(name, variant):
    return f"{BASE}/{name}__{variant}"


def base_overrides():
    """The same for every stage: the Manila data, an LSTM-only model, GeoCP off, cross-validation as USE_CV says, the transfer settings."""
    kv = {**pinned_stations(), "data.split.cv_assignment": "by_station", "geocp.enabled": "false", "model.spatial_cnn.enabled": "false", "train.cv.enabled": "true" if USE_CV else "false",
          "data.paths.master_raster": f"../data/Morphology/{RASTER}",
          "data.paths.sensor_locations_csv": f"../data/Combined Dataset/{ARM}/locations.csv",
          "data.paths.sensor_csv_dir": f"../data/Combined Dataset/{ARM}/sensors",
          "data.paths.processed_dir": f"runs/{ARM}/processed_nogeocp", "data.paths.pretrained_dir": PRETRAINED,
          "transfer.epochs": PRETRAIN_EPOCHS, "transfer.exclude_months": json.dumps(list(EXCLUDE_MONTHS))}
    return kv


def pinned_stations():
    """The fixed test/calibration stations as overrides (so a longer or updated dataset is scored on the same stations); [] when PIN_PARTITION is None."""
    if not PIN_PARTITION:
        return {}
    d = json.loads((REPO / PIN_PARTITION).read_text())
    return {"data.split.test_stations": json.dumps(sorted(d["test"])), "data.split.calibration_stations": json.dumps(sorted(d["calibration"]))}


def overrides(extra_kv=None, extra=()):
    kv = {**base_overrides(), **(extra_kv or {})}
    return [x for k, v in kv.items() for x in ("--override", f"{k}={v}")] + list(extra)


def job_kv(name, variant):
    """The overrides of one fine-tuning job: where it keeps its runs, and what it starts from."""
    d = job_dir(name, variant)
    kv = {"data.paths.checkpoints_dir": f"{d}/checkpoints", "data.paths.snapshots_dir": f"{d}/snapshots", "data.paths.tuning_dir": f"{d}/tuning",
          "train.final_epochs": "null" if USE_CV else EPOCHS}          # with cross-validation the best epochs of the folds set the final fit
    if variant != "scratch":
        kv["train.init_from"] = f"{PRETRAINED}/{name}/pretrained_lstm.pt"
    if variant == "head_only":
        kv["train.freeze"] = json.dumps(["temporal"])               # the pretrained LSTM stays as it is; only the fusion head learns
    if variant == "zero_shot":
        kv.update({"train.final_epochs": 1, "train.optimizer.lr": 0.0, "train.cv.enabled": "false"})     # one pass with learning rate 0: the weights do not move (nothing to cross-validate)
    return kv


def run(stage, extra_kv=None, *flags):
    """One pipeline stage in its own process. The stage is stopped when the time budget is used up (exit code 124 -> OutOfTime)."""
    if left_h() < MIN_STEP_H:
        raise OutOfTime(f"before {stage}")
    budget = ["--override", f"train.runtime.max_hours={left_h():.3f}"]
    cmd = [sys.executable, "-u", "run_pipeline.py", "--stage", stage, *overrides(extra_kv), *budget, *flags]
    print("$", " ".join(cmd), flush=True)
    t = time.time()
    code = subprocess.run(cmd, cwd=MODEL).returncode
    if code == 124:
        raise OutOfTime(f"during {stage}")
    if code != 0:
        raise subprocess.CalledProcessError(code, cmd)
    print(f"-- {stage}: {time.time() - t:.0f} s", flush=True)


def runs_of(name, variant):
    """Finished training runs of one job (run_info.json says completed), oldest first."""
    out = []
    for info in sorted((MODEL / job_dir(name, variant) / "checkpoints").glob("*/run_info.json")):
        if json.loads(info.read_text(encoding="utf-8")).get("status") == "completed":
            out.append(info.parent.name)
    return out


def have(*parts):
    return MODEL.joinpath(*parts).exists()


def status():
    """A line per job: what is finished."""
    rows = []
    for name, variant in jobs():
        runs = runs_of(name, variant)
        rows.append({"checkpoint": name, "variant": variant,
                     "pretrained": "n/a" if variant == "scratch" else "done" if have(PRETRAINED, name, "pretrained_lstm.pt") else "-",
                     "trained": "done" if runs else "-",
                     "reported": "done" if runs and have(job_dir(name, variant), "snapshots", runs[-1], "metrics_summary.json") else "-"})
    print(pd.DataFrame(rows).to_string(index=False) if rows else "no jobs yet (no source cities found?)")
    print(f"time used: {(time.time() - T0) / 3600:.1f} h of {TIME_BUDGET_H} h")


check_settings()
print("settings ok | dataset:", ARM, "| cross-validation:", USE_CV, "| checkpoints:", list(groups()), "| variants:", VARIANTS, "| repository:", REPO)

## 3. Get the repository, the data and the libraries
On Kaggle: a saved repository attached as an Input is restored (runs and all), otherwise the code is cloned from GitHub. Your data dataset must hold `Morphology/`, `Combined Dataset/<ARM>/` and `Other Countries/`.

In [ ]:
check_settings()


def find_saved_repo():
    """A repository saved by an earlier session: any attached input that contains model/run_pipeline.py."""
    if INPUT.exists():
        for p in INPUT.glob("**/model/run_pipeline.py"):
            return p.parent.parent
    return None


def clone(into):
    subprocess.run(["git", "clone", "--depth", "1", GITHUB, str(into)], check=True)


if ON_KAGGLE:
    if not (MODEL / "run_pipeline.py").is_file():
        saved = find_saved_repo()
        if saved is not None:
            shutil.copytree(saved, REPO, dirs_exist_ok=True)
            n = len(list((MODEL / "runs").glob("*/checkpoints*/*/run_info.json"))) if (MODEL / "runs").exists() else 0
            print(f"restored the repository from {saved} ({n} saved training run(s) found)")
        else:
            clone(REPO)
            print("cloned the repository from GitHub (no saved repository attached)")
    else:
        print("the repository is already in /kaggle/working")
    if REFRESH_CODE:                     # the latest code from GitHub; the saved runs and data are left as they are
        fresh = REPO.parent / "_fresh_code"
        shutil.rmtree(fresh, ignore_errors=True)
        clone(fresh)
        shutil.copytree(fresh / "model", MODEL, dirs_exist_ok=True, ignore=shutil.ignore_patterns("runs", "__pycache__", ".env"))
        shutil.rmtree(fresh, ignore_errors=True)
        print("code refreshed from GitHub")

    raster = REPO / "data" / "Morphology" / RASTER
    if not raster.is_file():             # the data is not in place yet: find it under DATA wherever the upload put it
        found = list(DATA.rglob(RASTER)) if DATA.exists() else []
        if found:
            src = found[0].parent.parent                      # the folder that holds Morphology/, Combined Dataset/, ...
            for part in (Path("Morphology"), Path("Boundaries"), Path("Combined Dataset") / ARM, Path("Other Countries")):
                if (src / part).is_dir():
                    shutil.copytree(src / part, REPO / "data" / part, dirs_exist_ok=True)
            print("data copied from", src)
        else:
            zips = list(DATA.rglob("*.zip")) if DATA.exists() else []
            if zips:                                          # Kaggle left an uploaded zip packed: its top-level `data/` folder lands in REPO
                import zipfile
                zipfile.ZipFile(zips[0]).extractall(REPO)
                print("data unzipped from", zips[0])

    need_pip = {"skopt": "scikit-optimize", "libpysal": "libpysal", "esda": "esda", "rasterio": "rasterio", "geopandas": "geopandas",
                "shapely": "shapely", "pyproj": "pyproj", "affine": "affine", "yaml": "pyyaml", "sklearn": "scikit-learn", "contextily": "contextily"}
    import importlib.util
    missing = [pip for mod, pip in need_pip.items() if importlib.util.find_spec(mod) is None]
    if missing:
        print("installing:", missing)
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)
    old_esda = "import inspect; from esda.moran import Moran_Local as M; raise SystemExit(0 if 'alternative' in inspect.signature(M.__init__).parameters else 1)"
    if subprocess.run([sys.executable, "-c", old_esda]).returncode != 0:      # the spatial-bias check needs the two-sided LISA option
        print("esda is too old: upgrading")
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "esda", "libpysal"], check=True)

# the checks that stop a run in its first seconds instead of after hours
problems = []
if not (MODEL / "run_pipeline.py").is_file():
    problems.append(f"{MODEL / 'run_pipeline.py'} is missing: no repository (attach a saved one, or check GITHUB and the internet switch)")
if not (REPO / "data" / "Morphology" / RASTER).is_file():
    problems.append(f"data/Morphology/{RASTER} is missing: put it in the Morphology folder of your data dataset (DATA = {DATA})")
if not (REPO / "data" / "Combined Dataset" / ARM / "locations.csv").is_file():
    problems.append(f"data/Combined Dataset/{ARM}/locations.csv is missing: check ARM and your data dataset")
if not source_cities():
    problems.append("data/Other Countries/<country>/<city>/source.json not found: run other-country-scrapers/scrapers.ipynb locally and put the data/Other Countries folder in your data dataset")
if problems:
    raise SystemExit("cannot start:\n  - " + "\n  - ".join(problems))
print("ready | source cities found:", source_cities() or "NONE (see the message below)")
status()

## 4. Preprocess Manila, pretrain, fine-tune, report
Order: preprocess Manila once (shared); then for each checkpoint: **pretrain** on its source cities (the transfer package scales them with Manila's scalers and keeps early stopping on held-out source stations),
then each variant: **train** on Manila (starting from the pretrained weights), **report** on the test stations. Every step is skipped if its output is there.
`mmd_top_<k>` pretrains on the `TOP_K` source stations closest to the Manila training stations; the ranking of all source stations is saved as `station_ranking.csv` beside each pretrained model.

In [ ]:
check_settings()
GROUP = groups()


def pretrain(name):
    cities = GROUP[name]
    extra_kv = {"transfer.top_k_stations": TOP_K if name.startswith("mmd_top") else "null"}
    flags = []
    for c in cities:
        flags += ["--source", c]
    run("pretrain", extra_kv, "--name", name, *flags)


def run_job(name, variant):
    print("=" * 78 + f"\n{name} / {variant}  ({left_h():.1f} h of the time budget left)\n" + "=" * 78)
    if variant != "scratch" and ("pretrain" in REDO or not have(PRETRAINED, name, "pretrained_lstm.pt")):
        pretrain(name)
    kv = job_kv(name, variant)
    if "train" in REDO or not runs_of(name, variant):
        run("train", kv, "--override", "train.resume=true")
    run_id = runs_of(name, variant)[-1]
    if "report" in REDO or not have(job_dir(name, variant), "snapshots", run_id, "metrics_summary.json"):
        run("report", kv, "--run-id", run_id)


try:
    if not have(f"runs/{ARM}/processed_nogeocp", "scalers.json") or "preprocess" in REDO:
        run("preprocess")
    else:
        print("skip preprocess: already done")
    for name, variant in jobs():
        run_job(name, variant)
    print("\nevery checkpoint is finished")
except OutOfTime as stop:
    print(f"\nTIME BUDGET REACHED ({stop}) after {(time.time() - T0) / 3600:.1f} h. Nothing is lost: finished steps are saved.")
    print("To continue: Save Version, then in a new session Add Input (this notebook's output + your data), and run all again.")
except Exception:
    traceback.print_exc()
    print("A step failed (everything finished so far is saved). Fix the cause above and run this cell again: finished steps are skipped.")
finally:
    status()

## 5. The comparison
Test R², RMSE and MAE of every checkpoint and variant on the **same Manila test stations**, and the change against `manila_only`. Read `finetune` against `manila_only` for the transfer; `zero_shot` for how much transfers without any Manila training.

With `USE_CV = True` the table also has the **cross-validation R²** on the Manila training stations (each station held out in turn, the pretrained weights loaded fresh in every fold) and, for each pool, in how many folds it beat `manila_only`. The pool is **chosen on that**, and its test R² is looked at once afterwards, so the choice does not use the test stations.

In [ ]:
try:
    need("VARIANTS")
    import matplotlib
    import matplotlib.pyplot as plt

    rows = []
    for name, variant in jobs():
        runs = runs_of(name, variant)
        if not runs:
            continue
        f = MODEL / job_dir(name, variant) / "snapshots" / runs[-1] / "metrics_summary.json"
        if not f.is_file():
            continue
        meas = json.loads(f.read_text(encoding="utf-8"))["accuracy"]["measures"]
        t, cvm = meas["test"], meas.get("cross_validation") or {}
        info = json.loads((MODEL / job_dir(name, variant) / "checkpoints" / runs[-1] / "run_info.json").read_text(encoding="utf-8"))
        folds = [c["metrics"]["r2"] for c in info.get("cv", [])]
        rows.append({"checkpoint": name, "variant": variant, "CV_R2": cvm.get("r2"), "test_R2": t["r2"], "test_RMSE": t["rmse"], "test_MAE": t["mae"],
                     "test_stations": t["stations"], "fold_R2": folds, "run": runs[-1]})
    table = pd.DataFrame(rows)
    if table.empty:
        print("no finished job with a report yet")
    else:
        base = table.loc[table["variant"] == "scratch", "test_R2"]
        table["R2 vs manila_only"] = table["test_R2"] - float(base.iloc[0]) if len(base) else np.nan
        pd.set_option("display.width", 220)
        base_cv = table.loc[table["variant"] == "scratch", "CV_R2"]
        base_folds = table.loc[table["variant"] == "scratch", "fold_R2"]
        if USE_CV and len(base_cv) and pd.notna(base_cv.iloc[0]):
            table["CV_R2 vs manila_only"] = table["CV_R2"] - float(base_cv.iloc[0])
            bf = base_folds.iloc[0]
            table["folds won"] = [("-" if r.variant == "scratch" or not r.fold_R2 or len(r.fold_R2) != len(bf) else f"{sum(a > b for a, b in zip(r.fold_R2, bf))}/{len(bf)}")
                                  for r in table.itertuples()]
        show = table.drop(columns=["fold_R2"])
        print(show.round(4).to_string(index=False))
        show.round(4).to_csv(OUT / "transfer_comparison.csv", index=False)
        if USE_CV and len(base_cv) and pd.notna(base_cv.iloc[0]):
            ft = table[table["variant"] == "finetune"].dropna(subset=["CV_R2"])
            if len(ft):
                best = ft.sort_values("CV_R2", ascending=False).iloc[0]
                print(f"\nSELECTED POOL (highest CV R2, the test stations are not used for the choice): {best.checkpoint}  CV R2 {best.CV_R2:.4f} vs manila_only {float(base_cv.iloc[0]):.4f} "
                      f"({best.CV_R2 - float(base_cv.iloc[0]):+.4f}), folds won {show.loc[best.name, 'folds won']}")
                print(f"its TEST R2 (looked at once, after choosing): {best.test_R2:.4f} vs manila_only {float(base.iloc[0]):.4f} ({best.test_R2 - float(base.iloc[0]):+.4f})")
                print("Integrate a pool only if it wins on CV R2 AND in most folds; a gain smaller than the spread between folds is noise.")
        print("written:", OUT / "transfer_comparison.csv")
        fig, ax = plt.subplots(figsize=(10, 4.8))
        order = table.sort_values("CV_R2" if USE_CV and table["CV_R2"].notna().any() else "test_R2")
        labels = [f"{r.checkpoint}\n{r.variant}" for r in order.itertuples()]
        colours = ["#8A9AA8" if r.variant == "scratch" else "#2B66B1" if r.variant == "finetune" else "#D2691E" if r.variant == "head_only" else "#7B6BB1" for r in order.itertuples()]
        ax.bar(range(len(order)), order["test_R2"], color=colours)
        ax.set_xticks(range(len(order)), labels, fontsize=6, rotation=0)
        lo = max(0.0, float(order["test_R2"].min()) - 0.05)
        ax.set_ylim(lo, float(order["test_R2"].max()) + 0.01)
        if len(base):
            ax.axhline(float(base.iloc[0]), color="#17212B", lw=1, ls="--")
            ax.text(0, float(base.iloc[0]), " manila_only", fontsize=7, va="bottom")
        ax.set_ylabel("test R2 (same Manila test stations)")
        ax.set_title("Transfer learning of the temporal side", fontsize=10, loc="left")
        fig.tight_layout()
        fig.savefig(OUT / "transfer_comparison.png", dpi=150)
        plt.show()
except Exception:
    traceback.print_exc()
    print("The comparison failed; each job's report is model/runs/<dataset>/tl/<checkpoint>__<variant>/snapshots/<run>/SUMMARY.md")